# PromptSentinel — Notebook 3
## Long Prompt & Human-Written Jailbreak Analysis

**Author:** Leesha Mogha

**Institution:** IMS Ghaziabad (University Course Campus)

**Project:** PromptSentinel (v4)

**Builds on:** [Prompt-Safety-Classifier](https://github.com/leeeshart/Prompt-Safety-Classifier)

---

### What this notebook does
Tests whether the v4 model maintains its performance on longer,
human-written jailbreaks where harmful content is buried in
creative text — the failure mode Notebook 2 could not test
because WildJailbreak prompts are synthetically generated.

### Research question answered here
**RQ2: Does performance drop on longer, human-written jailbreaks?**

In [1]:
!pip install datasets pandas numpy scikit-learn matplotlib seaborn -q

from datasets import load_dataset
from huggingface_hub import login
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    recall_score,
    precision_score,
    f1_score
)
import warnings
warnings.filterwarnings('ignore')

login()

print("Libraries loaded!")

Libraries loaded!


---
## Section 2: Load Data

Two data sources needed for this notebook:

1. **Combined dataset** (from Notebook 1) — used to train the v4 model
2. **TrustAIRLab** — reloaded separately as the human-written jailbreak test set

We keep TrustAIRLab completely out of training so it serves
as a clean out-of-distribution test. This is the

In [2]:
# Upload combined_dataset_final.csv from Notebook 1
from google.colab import files
import pandas as pd

print("Please upload compressed_data.csv.gz")
uploaded = files.upload()

df_train = pd.read_csv('compressed_data.csv.gz')

print(f"Combined dataset loaded: {len(df_train):,} prompts")
print(df_train['label'].value_counts())
print()
print("Sources:")
print(df_train['source'].value_counts())

Please upload compressed_data.csv.gz


Saving compressed_data.csv.gz to compressed_data.csv (1).gz
Combined dataset loaded: 116,202 prompts
label
safe      63142
unsafe    53060
Name: count, dtype: int64

Sources:
source
wildjailbreak    100099
trustairlab        6142
toxicchat          4981
qualifire          4980
Name: count, dtype: int64


### Removing TrustAIRLab from training data

TrustAIRLab will be used as the held-out human-written test set.
It must be completely excluded from training to avoid data leakage.

In [5]:
# Remove TrustAIRLab from training data
df_train = df_train[df_train['source'] != 'trustairlab'].reset_index(drop=True)

print(f"Training set after removing TrustAIRLab: {len(df_train):,} prompts")
print(df_train['label'].value_counts())
print()
print("Sources remaining in training:")
print(df_train['source'].value_counts())

Training set after removing TrustAIRLab: 110,060 prompts
label
safe      57653
unsafe    52407
Name: count, dtype: int64

Sources remaining in training:
source
wildjailbreak    100099
toxicchat          4981
qualifire          4980
Name: count, dtype: int64


### TrustAIRLab — Human-written jailbreak test set

Reloading TrustAIRLab separately and keeping it completely
outside of training. These are real human-written jailbreaks
collected from Reddit and Discord — not synthetic wraps.

In [6]:
from datasets import load_dataset
# Load TrustAIRLab as the held-out human-written test set
jailbreak = load_dataset(
    'TrustAIRLab/in-the-wild-jailbreak-prompts',
    'jailbreak_2023_05_07',
    split='train'
)
regular = load_dataset(
    'TrustAIRLab/in-the-wild-jailbreak-prompts',
    'regular_2023_05_07',
    split='train'
)

df_unsafe = jailbreak.to_pandas()[['prompt']]
df_safe   = regular.to_pandas()[['prompt']]

df_unsafe['label']  = 'unsafe'
df_safe['label']    = 'safe'

df_test_human = pd.concat([df_unsafe, df_safe], ignore_index=True)
df_test_human = df_test_human.dropna()
df_test_human = df_test_human[df_test_human['prompt'].str.strip() != '']

print(f"TrustAIRLab test set loaded: {len(df_test_human):,} prompts")
print(df_test_human['label'].value_counts())

TrustAIRLab test set loaded: 6,387 prompts
label
safe      5721
unsafe     666
Name: count, dtype: int64
